<a href="https://colab.research.google.com/github/gabriela0402/pathfolio-AI/blob/main/notebooks/00_environment_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pathfolio — Verificação e ingestão inicial

Este notebook será usado para testar a leitura de textos digitados,
arquivos PDF e arquivos DOCX.


## 1. Preparação do ambiente

Nesta etapa, o Pathfolio receberá informações escritas manualmente e as transformará em uma estrutura padronizada.


In [1]:
from datetime import datetime, timezone

print("Ambiente do Pathfolio funcionando!")


Ambiente do Pathfolio funcionando!


In [2]:
def create_text_source(
    content: str,
    source_name: str,
    category: str = "manual_text"
) -> dict:
    """
    Converte um texto digitado pelo usuário em uma fonte padronizada.
    """

    cleaned_content = content.strip()

    if not cleaned_content:
        raise ValueError("O conteúdo não pode estar vazio.")

    return {
        "source_type": "manual_text",
        "source_name": source_name,
        "category": category,
        "content": cleaned_content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": None
        }
    }


In [3]:
manual_source = create_text_source(
    content="""
    Participei de uma iniciação científica envolvendo a comparação de modelos
    de inteligência artificial aplicados à correção de redações. Trabalhei com
    uma base de dados extensa, análise de resultados e comparação de métricas.
    """,
    source_name="Experiência acadêmica",
    category="academic_experience"
)

manual_source


{'source_type': 'manual_text',
 'source_name': 'Experiência acadêmica',
 'category': 'academic_experience',
 'content': 'Participei de uma iniciação científica envolvendo a comparação de modelos\n    de inteligência artificial aplicados à correção de redações. Trabalhei com\n    uma base de dados extensa, análise de resultados e comparação de métricas.',
 'metadata': {'created_at': '2026-11-07T01:22:14.514805+00:00',
  'page': None,
  'file_name': None}}

In [4]:
assert manual_source["source_type"] == "manual_text"
assert manual_source["source_name"] == "Experiência acadêmica"
assert manual_source["category"] == "academic_experience"
assert len(manual_source["content"]) > 0
assert manual_source["metadata"]["page"] is None

print("Teste de texto manual aprovado.")


Teste de texto manual aprovado.


In [5]:
try:
    create_text_source(
        content="   ",
        source_name="Texto vazio"
    )
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: O conteúdo não pode estar vazio.


## 2. Leitura de documentos

Nesta etapa, o Pathfolio será capaz de ler arquivos PDF e DOCX.

Cada documento será convertido para uma ou mais fontes padronizadas,
preservando o nome do arquivo e, quando possível, a página de origem.


In [6]:
!pip -q install pypdf python-docx


In [7]:
from pathlib import Path

from pypdf import PdfReader
from docx import Document
from google.colab import files


In [8]:
def read_pdf(file_path: str) -> list[dict]:
    """
    Extrai o texto de um PDF página por página.
    """

    reader = PdfReader(file_path)
    sources = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        if not text:
            continue

        sources.append({
            "source_type": "pdf",
            "source_name": Path(file_path).name,
            "category": "uploaded_document",
            "content": text,
            "metadata": {
                "created_at": datetime.now(timezone.utc).isoformat(),
                "page": page_number,
                "file_name": Path(file_path).name
            }
        })

    return sources


In [9]:
def read_docx(file_path: str) -> list[dict]:
    """
    Extrai o texto dos parágrafos de um arquivo DOCX.
    """

    document = Document(file_path)
    paragraphs = []

    for paragraph in document.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    content = "\n".join(paragraphs).strip()

    if not content:
        return []

    return [{
        "source_type": "docx",
        "source_name": Path(file_path).name,
        "category": "uploaded_document",
        "content": content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": Path(file_path).name
        }
    }]


In [10]:
def read_document(file_path: str) -> list[dict]:
    """
    Lê um PDF ou DOCX usando o leitor correspondente.
    """

    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return read_pdf(file_path)

    if extension == ".docx":
        return read_docx(file_path)

    raise ValueError(
        f"Formato não suportado: {extension}. "
        "Envie um arquivo PDF ou DOCX."
    )


In [11]:
uploaded_files = files.upload()


Saving 01_curriculo_candidato_ficticio (1).pdf to 01_curriculo_candidato_ficticio (1).pdf
Saving 03_descricao_projeto_ficticio.docx to 03_descricao_projeto_ficticio (1).docx
Saving 02_relatorio_projeto_ficticio.pdf to 02_relatorio_projeto_ficticio (1).pdf


In [12]:
uploaded_sources = []

for file_name in uploaded_files.keys():
    document_sources = read_document(file_name)
    uploaded_sources.extend(document_sources)

print(f"Fontes extraídas: {len(uploaded_sources)}")

for source in uploaded_sources:
    print(f"Arquivo: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Página: {source['metadata']['page']}")
    print(f"Caracteres: {len(source['content'])}")
    print("-" * 50)


Fontes extraídas: 4
Arquivo: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Página: 1
Caracteres: 907
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Página: 2
Caracteres: 522
--------------------------------------------------
Arquivo: 03_descricao_projeto_ficticio (1).docx
Tipo: docx
Página: None
Caracteres: 807
--------------------------------------------------
Arquivo: 02_relatorio_projeto_ficticio (1).pdf
Tipo: pdf
Página: 1
Caracteres: 870
--------------------------------------------------


In [13]:
all_sources = [manual_source] + uploaded_sources

print(f"Total de fontes: {len(all_sources)}")

for index, source in enumerate(all_sources, start=1):
    print(
        f"{index}. "
        f"{source['source_name']} | "
        f"{source['source_type']} | "
        f"{len(source['content'])} caracteres"
    )


Total de fontes: 5
1. Experiência acadêmica | manual_text | 231 caracteres
2. 01_curriculo_candidato_ficticio (1).pdf | pdf | 907 caracteres
3. 01_curriculo_candidato_ficticio (1).pdf | pdf | 522 caracteres
4. 03_descricao_projeto_ficticio (1).docx | docx | 807 caracteres
5. 02_relatorio_projeto_ficticio (1).pdf | pdf | 870 caracteres


In [14]:
for source in all_sources:
    preview = source["content"][:300].replace("\n", " ")

    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Prévia: {preview}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos     de inteligência artificial aplicados à correção de redações. Trabalhei com     uma base de dados extensa, análise de resultados e comparação de métricas.
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Prévia: Currículo de Demonstração Marina Oliveira Candidata fictícia para testes do Pathfolio Objetivo Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações Inteligentes. Formação Graduação em Sistemas de Informação Universidade Exemplo — conclusão prevista: 2027 
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Prévia: Currículo de Demonstração — continuação Habilidades Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de Dados · Git · GitHub · SQL básico Cursos e atividades  Introdução a Machine Learning — curso livre.  Python para análise de dados

In [15]:
assert len(all_sources) >= 1

for source in all_sources:
    assert source["content"].strip()
    assert source["source_name"]
    assert source["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in source
    assert "file_name" in source["metadata"]

print("Testes de ingestão de documentos aprovados.")


Testes de ingestão de documentos aprovados.


In [16]:
try:
    read_document("arquivo.txt")
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: Formato não suportado: .txt. Envie um arquivo PDF ou DOCX.


## 3. Limpeza e divisão dos conteúdos

Nesta etapa, os conteúdos extraídos de textos manuais, PDFs e DOCX serão limpos e divididos em trechos menores.

Esses trechos serão utilizados posteriormente para criar embeddings e realizar busca semântica.


In [17]:
import re


def clean_text(text: str) -> str:
    """
    Limpa espaços e quebras de linha desnecessárias.
    """

    if not isinstance(text, str):
        raise TypeError("O conteúdo precisa ser uma string.")

    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [18]:
dirty_text = """
    Python     e análise de dados.


    Experiência com modelos de linguagem.
"""

cleaned_text = clean_text(dirty_text)

print(cleaned_text)


Python e análise de dados.

 Experiência com modelos de linguagem.


In [19]:
assert "     " not in cleaned_text
assert cleaned_text.strip() == cleaned_text

print("Teste de limpeza aprovado.")


Teste de limpeza aprovado.


In [20]:
cleaned_sources = []

for source in all_sources:
    cleaned_source = source.copy()
    cleaned_source["content"] = clean_text(source["content"])
    cleaned_sources.append(cleaned_source)

print(f"Fontes processadas: {len(cleaned_sources)}")


Fontes processadas: 5


In [21]:
for source in cleaned_sources:
    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Caracteres: {len(source['content'])}")
    print(f"Prévia: {source['content'][:200]}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Caracteres: 225
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados 
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Caracteres: 907
Prévia: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações
Intelig
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Caracteres: 522
Prévia: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e ativid
Fonte: 03_descricao_projeto_ficticio (1).docx
Tipo: docx
Caracteres: 807
Prévia: Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estu

In [22]:
def split_text(
    text: str,
    chunk_size: int = 800,
    chunk_overlap: int = 120
) -> list[str]:
    """
    Divide um texto em trechos com sobreposição.

    A sobreposição ajuda a preservar o contexto entre dois trechos.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size precisa ser maior que zero.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap não pode ser negativo.")

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap precisa ser menor que chunk_size."
        )

    text = clean_text(text)

    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)
    step = chunk_size - chunk_overlap

    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start += step

    return chunks


In [23]:
test_text = (
    "Python é uma linguagem utilizada em análise de dados. "
    "Machine Learning permite construir modelos preditivos. "
    "Modelos de linguagem podem processar textos e gerar respostas. "
    "A avaliação dos resultados é importante para entender limitações."
)

test_chunks = split_text(
    test_text,
    chunk_size=100,
    chunk_overlap=20
)

for index, chunk in enumerate(test_chunks):
    print(f"Trecho {index}:")
    print(chunk)
    print("-" * 50)


Trecho 0:
Python é uma linguagem utilizada em análise de dados. Machine Learning permite construir modelos pre
--------------------------------------------------
Trecho 1:
onstruir modelos preditivos. Modelos de linguagem podem processar textos e gerar respostas. A avalia
--------------------------------------------------
Trecho 2:
respostas. A avaliação dos resultados é importante para entender limitações.
--------------------------------------------------


In [24]:
assert len(test_chunks) > 1
assert all(len(chunk) > 0 for chunk in test_chunks)

print("Teste de divisão de texto aprovado.")


Teste de divisão de texto aprovado.


In [25]:
def create_chunks(
    sources: list[dict],
    chunk_size: int = 800,
    chunk_overlap: int = 120
) -> list[dict]:
    """
    Divide todas as fontes em trechos e preserva seus metadados.
    """

    chunks = []
    chunk_counter = 0

    for source in sources:
        source_chunks = split_text(
            source["content"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_index, content in enumerate(source_chunks):
            chunks.append({
                "chunk_id": f"chunk-{chunk_counter:04d}",
                "content": content,
                "source_name": source["source_name"],
                "source_type": source["source_type"],
                "category": source["category"],
                "metadata": {
                    "page": source["metadata"].get("page"),
                    "file_name": source["metadata"].get("file_name"),
                    "chunk_index": chunk_index,
                    "source_created_at": source["metadata"].get(
                        "created_at"
                    )
                }
            })

            chunk_counter += 1

    return chunks


In [26]:
chunks = create_chunks(cleaned_sources)

print(f"Total de chunks criados: {len(chunks)}")


Total de chunks criados: 8


In [27]:
for chunk in chunks[:5]:
    print(f"ID: {chunk['chunk_id']}")
    print(f"Fonte: {chunk['source_name']}")
    print(f"Tipo: {chunk['source_type']}")
    print(f"Página: {chunk['metadata']['page']}")
    print(f"Índice: {chunk['metadata']['chunk_index']}")
    print(f"Conteúdo: {chunk['content'][:300]}")
    print("=" * 80)


ID: chunk-0000
Fonte: Experiência acadêmica
Tipo: manual_text
Página: None
Índice: 0
Conteúdo: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados e comparação de métricas.
ID: chunk-0001
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Página: 1
Índice: 0
Conteúdo: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações
Inteligentes.
Formação
Graduação em Sistemas de Informação
Universidade Exemplo — conclusão prevista: 2027

ID: chunk-0002
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Página: 1
Índice: 1
Conteúdo: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de P

In [28]:
assert len(chunks) > 0

for chunk in chunks:
    assert chunk["chunk_id"]
    assert chunk["content"]
    assert chunk["source_name"]
    assert chunk["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in chunk
    assert "page" in chunk["metadata"]

print("Testes de chunks e metadados aprovados.")


Testes de chunks e metadados aprovados.


In [29]:
from collections import Counter

chunks_per_source = Counter(
    chunk["source_name"]
    for chunk in chunks
)

for source_name, quantity in chunks_per_source.items():
    print(f"{source_name}: {quantity} chunks")


Experiência acadêmica: 1 chunks
01_curriculo_candidato_ficticio (1).pdf: 3 chunks
03_descricao_projeto_ficticio (1).docx: 2 chunks
02_relatorio_projeto_ficticio (1).pdf: 2 chunks


In [30]:
def keyword_search(
    chunks: list[dict],
    query: str
) -> list[dict]:
    """
    Retorna chunks que contêm todas as palavras relevantes da busca.
    """

    query_terms = [
        term.lower()
        for term in re.findall(r"\w+", query)
        if len(term) > 2
    ]

    results = []

    for chunk in chunks:
        content = chunk["content"].lower()

        if all(term in content for term in query_terms):
            results.append(chunk)

    return results


In [31]:
results = keyword_search(
    chunks,
    "Python análise de dados"
)

print(f"Resultados encontrados: {len(results)}")

for result in results:
    print(result["source_name"])
    print(result["content"])
    print("-" * 80)


Resultados encontrados: 2
01_curriculo_candidato_ficticio (1).pdf
Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
--------------------------------------------------------------------------------
02_relatorio_projeto_ficticio (1).pdf
Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade, facilitando a análise por professores.
A

In [32]:
import json

chunks_output_path = "pathfolio_chunks_debug.json"

with open(chunks_output_path, "w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

print(f"Arquivo salvo: {chunks_output_path}")


Arquivo salvo: pathfolio_chunks_debug.json


## 4. Embeddings e busca semântica

Nesta etapa, cada chunk será convertido em um vetor numérico chamado embedding.

Chunks semanticamente parecidos ficarão próximos no espaço vetorial. Assim,
o Pathfolio poderá encontrar evidências relacionadas a uma vaga mesmo quando
as palavras utilizadas forem diferentes.


In [33]:
!pip -q install sentence-transformers scikit-learn


In [34]:
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [35]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print(f"Modelo carregado: {embedding_model_name}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:140: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo carregado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [36]:
chunk_texts = [
    chunk["content"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Quantidade de embeddings: {len(chunk_embeddings)}")
print(f"Dimensão de cada embedding: {chunk_embeddings.shape[1]}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Quantidade de embeddings: 8
Dimensão de cada embedding: 384


In [37]:
assert len(chunk_embeddings) == len(chunks)
assert chunk_embeddings.ndim == 2
assert chunk_embeddings.shape[1] > 0
assert np.isfinite(chunk_embeddings).all()

print("Teste de embeddings aprovado.")


Teste de embeddings aprovado.


In [38]:
embedding_norms = np.linalg.norm(chunk_embeddings, axis=1)

print(embedding_norms[:5])


[1.         0.99999994 1.         1.         0.99999994]


In [39]:
def semantic_search(
    query: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5
) -> list[dict]:
    """
    Retorna os chunks semanticamente mais próximos da consulta.
    """

    if not query or not query.strip():
        raise ValueError("A consulta não pode estar vazia.")

    if len(chunks) != len(chunk_embeddings):
        raise ValueError(
            "A quantidade de chunks precisa ser igual à quantidade de embeddings."
        )

    if top_k <= 0:
        raise ValueError("top_k precisa ser maior que zero.")

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]
    ranked_indices = ranked_indices[:min(top_k, len(chunks))]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        result = chunks[index].copy()
        result["similarity_score"] = float(similarities[index])
        result["rank"] = rank
        results.append(result)

    return results


In [40]:
query = """
Experiência com Python, análise de dados e avaliação de modelos de inteligência artificial.
"""

semantic_results = semantic_search(
    query=query,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)


In [41]:
for result in semantic_results:
    print(f"Rank: {result['rank']}")
    print(f"Similaridade: {result['similarity_score']:.4f}")
    print(f"Fonte: {result['source_name']}")
    print(f"Tipo: {result['source_type']}")
    print(f"Página: {result['metadata']['page']}")
    print(f"Conteúdo: {result['content']}")
    print("=" * 100)


Rank: 1
Similaridade: 0.6570
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Página: 2
Conteúdo: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
Rank: 2
Similaridade: 0.6483
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Tipo: pdf
Página: 1
Conteúdo: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
Rank: 3
Similaridade: 0.6235
Fonte: E

In [42]:
query_semantic = """
Vivência em experimentos com modelos computacionais capazes de trabalhar com linguagem humana.
"""

semantic_results_2 = semantic_search(
    query=query_semantic,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in semantic_results_2:
    print(
        f"Rank {result['rank']} | "
        f"Score {result['similarity_score']:.4f} | "
        f"{result['source_name']}"
    )
    print(result["content"][:500])
    print("-" * 80)


Rank 1 | Score 0.6798 | 01_curriculo_candidato_ficticio (1).pdf
s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
--------------------------------------------------------------------------------
Rank 2 | Score 0.6141 | 01_curriculo_candidato_ficticio (1).pdf
Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento
--------------------------------------------------------------

In [43]:
job_description = """
Estágio em Inteligência Artificial.

A pessoa candidata irá apoiar projetos de análise de dados e desenvolvimento
de soluções baseadas em modelos de linguagem. É desejável conhecimento em
Python, machine learning, processamento de linguagem natural, avaliação de
modelos e documentação de experimentos.
"""


In [44]:
job_results = semantic_search(
    query=job_description,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in job_results:
    print(f"Rank: {result['rank']}")
    print(f"Score: {result['similarity_score']:.4f}")
    print(f"Fonte: {result['source_name']}")
    print(f"Página: {result['metadata']['page']}")
    print(f"Trecho: {result['content']}")
    print("=" * 100)


Rank: 1
Score: 0.7510
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Página: 2
Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
Rank: 2
Score: 0.7422
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Página: 1
Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
Rank: 3
Score: 0.7071
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Página

In [45]:
def format_evidence(results: list[dict]) -> list[dict]:
    """
    Formata os resultados para serem apresentados como evidências.
    """

    evidences = []

    for result in results:
        evidences.append({
            "evidence_id": result["chunk_id"],
            "rank": result["rank"],
            "similarity_score": result["similarity_score"],
            "source_name": result["source_name"],
            "source_type": result["source_type"],
            "page": result["metadata"].get("page"),
            "content": result["content"]
        })

    return evidences


In [46]:
evidences = format_evidence(job_results)

for evidence in evidences:
    print(evidence)


{'evidence_id': 'chunk-0003', 'rank': 1, 'similarity_score': 0.7510434985160828, 'source_name': '01_curriculo_candidato_ficticio (1).pdf', 'source_type': 'pdf', 'page': 2, 'content': 'Currículo de Demonstração — continuação\nHabilidades\nPython · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de\nDados · Git · GitHub · SQL básico\nCursos e atividades\n\x7f\nIntrodução a Machine Learning — curso livre.\n\x7f\nPython para análise de dados — curso livre.\n\x7f\nParticipação em grupo de estudos sobre Inteligência Artificial Generativa.\nInformações adicionais\nInteresse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e\ndesenvolvimento de produtos digitais.'}
{'evidence_id': 'chunk-0002', 'rank': 2, 'similarity_score': 0.7422292828559875, 'source_name': '01_curriculo_candidato_ficticio (1).pdf', 'source_type': 'pdf', 'page': 1, 'content': 's\nAssistente de estudos com processamento de linguagem natural\nProjeto experimental

In [47]:
def filter_by_similarity(
    results: list[dict],
    minimum_score: float = 0.35
) -> list[dict]:
    """
    Mantém somente resultados acima da similaridade mínima.
    """

    return [
        result
        for result in results
        if result["similarity_score"] >= minimum_score
    ]


In [48]:
filtered_results = filter_by_similarity(
    job_results,
    minimum_score=0.35
)

print(f"Resultados antes do filtro: {len(job_results)}")
print(f"Resultados depois do filtro: {len(filtered_results)}")


Resultados antes do filtro: 5
Resultados depois do filtro: 5


In [49]:
unrelated_results = semantic_search(
    query="Experiência profissional em engenharia mecânica e manutenção industrial.",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in unrelated_results:
    print(
        f"{result['similarity_score']:.4f} | "
        f"{result['source_name']} | "
        f"{result['content'][:150]}"
    )


0.2714 | 03_descricao_projeto_ficticio (1).docx | Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de 
0.2398 | 01_curriculo_candidato_ficticio (1).pdf | Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artifici
0.2091 | 02_relatorio_projeto_ficticio (1).pdf | Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técn
0.2079 | Experiência acadêmica | Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com

0.2036 | 01_curriculo_candidato_ficticio (1).pdf | s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de


In [50]:
import json

semantic_results_path = "pathfolio_semantic_search_debug.json"

with open(semantic_results_path, "w", encoding="utf-8") as file:
    json.dump(
        job_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Arquivo salvo: {semantic_results_path}")


Arquivo salvo: pathfolio_semantic_search_debug.json


In [51]:
assert len(job_results) > 0
assert len(job_results) <= 5

for index, result in enumerate(job_results, start=1):
    assert result["rank"] == index
    assert 0 <= result["similarity_score"] <= 1
    assert result["content"]
    assert result["source_name"]

for first, second in zip(job_results, job_results[1:]):
    assert first["similarity_score"] >= second["similarity_score"]

print("Testes de busca semântica aprovados.")


Testes de busca semântica aprovados.


## 5. Análise de competências e lacunas

Nesta etapa, o Pathfolio analisará uma descrição de vaga e verificará
quais competências possuem evidências nas informações fornecidas pelo candidato.

A aplicação não deverá considerar uma competência como comprovada apenas
porque ela aparece na descrição da vaga. Ela precisará estar relacionada
a algum trecho recuperado das fontes do candidato.


In [52]:
job_description = """
Estágio em Inteligência Artificial

A pessoa candidata irá apoiar projetos de análise de dados e desenvolvimento
de soluções baseadas em modelos de linguagem.

Requisitos:
- Conhecimento em Python;
- Noções de machine learning;
- Interesse em modelos de linguagem;
- Capacidade de analisar dados;
- Boa comunicação para apresentar resultados.

Diferenciais:
- Experiência com processamento de linguagem natural;
- Experiência com avaliação de modelos;
- Participação em projetos de pesquisa.
"""


In [53]:
skill_catalog = [
    "Python",
    "Machine Learning",
    "Análise de dados",
    "Modelos de linguagem",
    "Processamento de Linguagem Natural",
    "Avaliação de modelos",
    "Pesquisa",
    "Git",
    "GitHub",
    "SQL",
    "React",
    "JavaScript",
    "Comunicação"
]


In [54]:
skill_aliases = {
    "Python": [
        "python"
    ],
    "Machine Learning": [
        "machine learning",
        "aprendizado de máquina",
        "ml"
    ],
    "Análise de dados": [
        "análise de dados",
        "analise de dados",
        "análise dos dados",
        "analise dos dados",
        "data analysis"
    ],
    "Modelos de linguagem": [
        "modelos de linguagem",
        "modelo de linguagem",
        "large language model",
        "large language models",
        "llm",
        "llms"
    ],
    "Processamento de Linguagem Natural": [
        "processamento de linguagem natural",
        "natural language processing",
        "nlp"
    ],
    "Avaliação de modelos": [
        "avaliação de modelos",
        "avaliacao de modelos",
        "avaliação de modelagem",
        "model evaluation"
    ],
    "Pesquisa": [
        "pesquisa",
        "iniciação científica",
        "iniciacao cientifica",
        "investigação",
        "investigacao"
    ],
    "Git": [
        "git"
    ],
    "GitHub": [
        "github"
    ],
    "SQL": [
        "sql"
    ],
    "React": [
        "react"
    ],
    "JavaScript": [
        "javascript",
        "java script"
    ],
    "Comunicação": [
        "comunicação",
        "comunicacao",
        "apresentação de resultados",
        "apresentacao de resultados"
    ]
}


In [55]:
def find_skills_in_text(
    text: str,
    skill_aliases: dict[str, list[str]]
) -> list[str]:
    """
    Identifica competências com base em aliases conhecidos.
    """

    normalized_text = text.lower()
    found_skills = []

    for skill, aliases in skill_aliases.items():
        for alias in aliases:
            if alias.lower() in normalized_text:
                found_skills.append(skill)
                break

    return found_skills


In [56]:
job_skills = find_skills_in_text(
    job_description,
    skill_aliases
)

print("Competências identificadas na vaga:")
for skill in job_skills:
    print(f"- {skill}")


Competências identificadas na vaga:
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa
- Comunicação


In [57]:
def build_skill_evidence_index(
    chunks: list[dict],
    skill_aliases: dict[str, list[str]]
) -> dict[str, list[dict]]:
    """
    Cria um índice que relaciona competências aos chunks que as mencionam.
    """

    evidence_index = {
        skill: []
        for skill in skill_aliases
    }

    for chunk in chunks:
        found_skills = find_skills_in_text(
            chunk["content"],
            skill_aliases
        )

        for skill in found_skills:
            evidence_index[skill].append({
                "chunk_id": chunk["chunk_id"],
                "source_name": chunk["source_name"],
                "source_type": chunk["source_type"],
                "page": chunk["metadata"].get("page"),
                "content": chunk["content"]
            })

    return evidence_index


In [58]:
skill_evidence_index = build_skill_evidence_index(
    chunks=chunks,
    skill_aliases=skill_aliases
)


In [59]:
for skill, evidences in skill_evidence_index.items():
    if evidences:
        print(f"\n{skill}: {len(evidences)} evidência(s)")

        for evidence in evidences[:2]:
            print(f"- Fonte: {evidence['source_name']}")
            print(f"- Página: {evidence['page']}")
            print(f"- Trecho: {evidence['content'][:250]}")



Python: 3 evidência(s)
- Fonte: 01_curriculo_candidato_ficticio (1).pdf
- Página: 1
- Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
- Fonte: 01_curriculo_candidato_ficticio (1).pdf
- Página: 2
- Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre

Machine Learning: 1 evidência(s)
- Fonte: 01_curriculo_candidato_ficticio (1).pdf
- Página: 2
- Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre

In [60]:
def analyze_skills_and_gaps(
    job_description: str,
    chunks: list[dict],
    skill_aliases: dict[str, list[str]]
) -> dict:
    """
    Analisa competências da vaga e verifica evidências no perfil.
    """

    job_skills = find_skills_in_text(
        job_description,
        skill_aliases
    )

    evidence_index = build_skill_evidence_index(
        chunks,
        skill_aliases
    )

    found_skills = []
    missing_skills = []
    skill_analysis = []

    for skill in job_skills:
        evidences = evidence_index.get(skill, [])

        if evidences:
            status = "found"
            found_skills.append(skill)
        else:
            status = "not_found"
            missing_skills.append(skill)

        skill_analysis.append({
            "skill": skill,
            "status": status,
            "evidence_count": len(evidences),
            "evidence": evidences[:3]
        })

    return {
        "job_skills": job_skills,
        "found_skills": found_skills,
        "missing_skills": missing_skills,
        "skill_analysis": skill_analysis
    }


In [61]:
skills_analysis = analyze_skills_and_gaps(
    job_description=job_description,
    chunks=chunks,
    skill_aliases=skill_aliases
)


In [62]:
print("COMPETÊNCIAS ENCONTRADAS")
print("=" * 60)

for skill in skills_analysis["found_skills"]:
    print(f"- {skill}")

print("\nCOMPETÊNCIAS NÃO ENCONTRADAS")
print("=" * 60)

for skill in skills_analysis["missing_skills"]:
    print(f"- {skill}")



COMPETÊNCIAS ENCONTRADAS
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa

COMPETÊNCIAS NÃO ENCONTRADAS
- Comunicação


In [63]:
def calculate_skill_coverage(
    found_skills: list[str],
    job_skills: list[str]
) -> float:
    """
    Calcula a proporção de competências da vaga com evidências.
    """

    if not job_skills:
        return 0.0

    return len(found_skills) / len(job_skills)


In [64]:
coverage = calculate_skill_coverage(
    found_skills=skills_analysis["found_skills"],
    job_skills=skills_analysis["job_skills"]
)

print(f"Cobertura de competências: {coverage:.2%}")


Cobertura de competências: 87.50%


In [65]:
def retrieve_skill_evidence(
    skill: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 3,
    minimum_score: float = 0.35
) -> list[dict]:
    """
    Recupera evidências semanticamente relacionadas a uma competência.
    """

    results = semantic_search(
        query=skill,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        model=model,
        top_k=top_k
    )

    return [
        result
        for result in results
        if result["similarity_score"] >= minimum_score
    ]


In [66]:
python_evidence = retrieve_skill_evidence(
    skill="Análise de dados",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=3
)

for evidence in python_evidence:
    print(f"Score: {evidence['similarity_score']:.4f}")
    print(f"Fonte: {evidence['source_name']}")
    print(f"Conteúdo: {evidence['content'][:400]}")
    print("-" * 80)


Score: 0.5523
Fonte: Experiência acadêmica
Conteúdo: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados e comparação de métricas.
--------------------------------------------------------------------------------
Score: 0.5411
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Conteúdo: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações
Inteligentes.
Formação
Graduação em Sistemas de Informação
Universidade Exemplo — conclusão prevista: 2027
Experiência acadêmica
Projeto de iniciação científica — Avaliação de modelos de linguagem

Comparaç
--------------------------------------------------------------------------------
Score: 0.4900
Fonte: 02_relatorio_projeto_ficticio (1).pdf
Conteúdo: dados, métricas de

In [67]:
def analyze_skill_hybrid(
    skill: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    skill_aliases: dict[str, list[str]],
    minimum_score: float = 0.35
) -> dict:
    """
    Analisa uma competência com evidência direta e semântica.

    Evidência direta confirma a competência.
    Evidência semântica isolada é marcada como candidata.
    """

    direct_evidence = []

    for chunk in chunks:
        found_skills = find_skills_in_text(
            chunk["content"],
            {skill: skill_aliases.get(skill, [])}
        )

        if skill in found_skills:
            direct_evidence.append({
                "chunk_id": chunk["chunk_id"],
                "source_name": chunk["source_name"],
                "source_type": chunk["source_type"],
                "page": chunk["metadata"].get("page"),
                "content": chunk["content"],
                "evidence_method": "keyword"
            })

    semantic_evidence = retrieve_skill_evidence(
        skill=skill,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        model=model,
        top_k=3,
        minimum_score=minimum_score
    )

    semantic_only_evidence = []
    direct_ids = {
        evidence["chunk_id"]
        for evidence in direct_evidence
    }

    for evidence in semantic_evidence:
        if evidence["chunk_id"] not in direct_ids:
            evidence_copy = evidence.copy()
            evidence_copy["evidence_method"] = "semantic_candidate"
            semantic_only_evidence.append(evidence_copy)

    if direct_evidence:
        status = "found"
        evidence_level = "direct"
    elif semantic_only_evidence:
        status = "candidate"
        evidence_level = "semantic_only"
    else:
        status = "not_found"
        evidence_level = "none"

    return {
        "skill": skill,
        "status": status,
        "evidence_level": evidence_level,
        "direct_evidence": direct_evidence[:5],
        "semantic_candidates": semantic_only_evidence[:5]
    }


In [68]:
def analyze_job_hybrid(
    job_description: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    skill_aliases: dict[str, list[str]],
    minimum_score: float = 0.35
) -> dict:
    """
    Analisa competências da vaga usando evidências diretas e semânticas.
    """

    job_skills = find_skills_in_text(
        job_description,
        skill_aliases
    )

    skill_results = []

    for skill in job_skills:
        result = analyze_skill_hybrid(
            skill=skill,
            chunks=chunks,
            chunk_embeddings=chunk_embeddings,
            model=model,
            skill_aliases=skill_aliases,
            minimum_score=minimum_score
        )

        skill_results.append(result)

    found_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "found"
    ]

    candidate_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "candidate"
    ]

    missing_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "not_found"
    ]

    return {
        "job_skills": job_skills,
        "found_skills": found_skills,
        "candidate_skills": candidate_skills,
        "missing_skills": missing_skills,
        "skill_results": skill_results,
        "coverage": calculate_skill_coverage(
            found_skills,
            job_skills
        )
    }


In [69]:
hybrid_analysis = analyze_job_hybrid(
    job_description=job_description,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    skill_aliases=skill_aliases,
    minimum_score=0.35
)


In [70]:
print(hybrid_analysis.keys())


dict_keys(['job_skills', 'found_skills', 'candidate_skills', 'missing_skills', 'skill_results', 'coverage'])


In [71]:
print(f"Cobertura: {hybrid_analysis['coverage']:.2%}")

print("\nEncontradas:")
for skill in hybrid_analysis["found_skills"]:
    print(f"- {skill}")

print("\nCandidatas:")
for skill in hybrid_analysis["candidate_skills"]:
    print(f"- {skill}")

print("\nNão encontradas:")
for skill in hybrid_analysis["missing_skills"]:
    print(f"- {skill}")


Cobertura: 87.50%

Encontradas:
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa

Candidatas:
- Comunicação

Não encontradas:


In [72]:
for result in hybrid_analysis["skill_results"]:
    print(f"Competência: {result['skill']}")
    print(f"Status: {result['status']}")
    print(f"Nível de evidência: {result['evidence_level']}")

    print("\nEvidências diretas:")
    for evidence in result["direct_evidence"]:
        print(f"- Fonte: {evidence['source_name']}")
        print(f"- Página: {evidence['page']}")
        print(f"- Trecho: {evidence['content'][:500]}")

    print("\nCandidatas semânticas:")
    for evidence in result["semantic_candidates"]:
        print(f"- Score: {evidence['similarity_score']:.4f}")
        print(f"- Fonte: {evidence['source_name']}")
        print(f"- Trecho: {evidence['content'][:500]}")

    print("=" * 100)


Competência: Python
Status: found
Nível de evidência: direct

Evidências diretas:
- Fonte: 01_curriculo_candidato_ficticio (1).pdf
- Página: 1
- Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
- Fonte: 01_curriculo_candidato_ficticio (1).pdf
- Página: 2
- Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento
- Fonte: 02_relatorio_projeto_ficticio (1).pdf


In [73]:
for result in hybrid_analysis["skill_results"]:
    print(
        f"{result['skill']} | "
        f"status={result['status']} | "
        f"evidência={result['evidence_level']} | "
        f"diretas={len(result['direct_evidence'])} | "
        f"candidatas={len(result['semantic_candidates'])}"
    )


Python | status=found | evidência=direct | diretas=3 | candidatas=0
Machine Learning | status=found | evidência=direct | diretas=1 | candidatas=2
Análise de dados | status=found | evidência=direct | diretas=1 | candidatas=3
Modelos de linguagem | status=found | evidência=direct | diretas=3 | candidatas=1
Processamento de Linguagem Natural | status=found | evidência=direct | diretas=4 | candidatas=0
Avaliação de modelos | status=found | evidência=direct | diretas=2 | candidatas=2
Pesquisa | status=found | evidência=direct | diretas=2 | candidatas=2
Comunicação | status=candidate | evidência=semantic_only | diretas=0 | candidatas=1


In [74]:
def show_missing_skills(analysis: dict) -> None:
    missing_skills = analysis["missing_skills"]

    if not missing_skills:
        print("Nenhuma lacuna foi identificada.")
        return

    print("Competências sem evidência direta:")
    for skill in missing_skills:
        print(f"- {skill}")


In [75]:
show_missing_skills(hybrid_analysis)


Nenhuma lacuna foi identificada.


In [76]:
def build_skill_report(analysis: dict) -> dict:
    return {
        "coverage": analysis["coverage"],
        "job_skills": analysis["job_skills"],
        "found_skills": analysis["found_skills"],
        "candidate_skills": analysis["candidate_skills"],
        "missing_skills": analysis["missing_skills"],
        "summary": {
            "total_skills": len(analysis["job_skills"]),
            "found_count": len(analysis["found_skills"]),
            "candidate_count": len(analysis["candidate_skills"]),
            "missing_count": len(analysis["missing_skills"])
        }
    }


skill_report = build_skill_report(hybrid_analysis)
skill_report


{'coverage': 0.875,
 'job_skills': ['Python',
  'Machine Learning',
  'Análise de dados',
  'Modelos de linguagem',
  'Processamento de Linguagem Natural',
  'Avaliação de modelos',
  'Pesquisa',
  'Comunicação'],
 'found_skills': ['Python',
  'Machine Learning',
  'Análise de dados',
  'Modelos de linguagem',
  'Processamento de Linguagem Natural',
  'Avaliação de modelos',
  'Pesquisa'],
 'candidate_skills': ['Comunicação'],
 'missing_skills': [],
 'summary': {'total_skills': 8,
  'found_count': 7,
  'candidate_count': 1,
  'missing_count': 0}}

In [77]:
import json

with open("pathfolio_skill_report.json", "w", encoding="utf-8") as file:
    json.dump(
        skill_report,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Relatório salvo.")


Relatório salvo.


In [78]:
all_analyzed_skills = (
    hybrid_analysis["found_skills"]
    + hybrid_analysis["candidate_skills"]
    + hybrid_analysis["missing_skills"]
)

assert set(all_analyzed_skills) == set(
    hybrid_analysis["job_skills"]
)

assert not (
    set(hybrid_analysis["found_skills"])
    & set(hybrid_analysis["missing_skills"])
)

assert not (
    set(hybrid_analysis["candidate_skills"])
    & set(hybrid_analysis["missing_skills"])
)

print("Teste de consistência da análise aprovado.")


Teste de consistência da análise aprovado.


## 6. Análise e geração estruturada com LLM

Nesta etapa, o Pathfolio utilizará um modelo generativo do Google Gemini
para analisar as evidências recuperadas e estruturar um currículo profissional.

A chave da API será lida pelo Secret `GEMINI_API_KEY2` do Google Colab.

A chave nunca será escrita diretamente no notebook, exibida na saída ou
enviada ao GitHub.

O modelo utilizado será definido na variável `gemini_model_name`.


In [79]:
!pip -q install -U google-genai


In [80]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY2")

if not GEMINI_API_KEY:
    raise ValueError(
        "O Secret GEMINI_API_KEY não foi encontrado no Colab."
    )

print("Chave do Gemini encontrada com segurança.")


Chave do Gemini encontrada com segurança.


In [81]:
from google import genai

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

gemini_model_name = "gemini-3.8-flash"

print(f"Modelo configurado: {gemini_model_name}")


Modelo configurado: gemini-3.8-flash


In [82]:
import time


def generate_with_retry(
    client,
    model,
    contents,
    config=None,
    attempts=3,
    wait_seconds=20
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(f"Tentativa {attempt}/{attempts}...")

            request_args = {
                "model": model,
                "contents": contents
            }

            if config is not None:
                request_args["config"] = config

            response = client.models.generate_content(
                **request_args
            )

            print("Resposta recebida.")
            return response

        except Exception as error:
            last_error = error
            print(f"Falhou: {type(error).__name__}")

            if attempt < attempts:
                print(f"Aguardando {wait_seconds} segundos...")
                time.sleep(wait_seconds)
                wait_seconds *= 2

    raise last_error


In [83]:
def prepare_evidence_context(
    analysis: dict,
    max_evidence_per_skill: int = 2
) -> str:
    sections = []

    for result in analysis["skill_results"]:
        skill = result["skill"]
        status = result["status"]
        evidence_level = result["evidence_level"]

        sections.append(f"Competência: {skill}")
        sections.append(f"Status: {status}")
        sections.append(f"Nível da evidência: {evidence_level}")

        direct_evidence = result.get("direct_evidence", [])

        if direct_evidence:
            sections.append("Evidências diretas:")

            for evidence in direct_evidence[:max_evidence_per_skill]:
                sections.append(
                    f"ID: {evidence['chunk_id']}\n"
                    f"Fonte: {evidence['source_name']}\n"
                    f"Página: {evidence.get('page')}\n"
                    f"Trecho: {evidence['content']}"
                )

        semantic_candidates = result.get(
            "semantic_candidates",
            []
        )

        if semantic_candidates:
            sections.append("Evidências candidatas semânticas:")

            for evidence in semantic_candidates[:max_evidence_per_skill]:
                sections.append(
                    f"ID candidato: {evidence['chunk_id']}\n"
                    f"Fonte: {evidence['source_name']}\n"
                    f"Página: {evidence.get('page')}\n"
                    f"Trecho candidato: {evidence['content']}"
                )

        sections.append("-" * 60)

    return "\n".join(sections)


In [84]:
evidence_context = prepare_evidence_context(
    hybrid_analysis
)

print(evidence_context)


Competência: Python
Status: found
Nível da evidência: direct
Evidências diretas:
ID: chunk-0002
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Página: 1
Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
ID: chunk-0003
Fonte: 01_curriculo_candidato_ficticio (1).pdf
Página: 2
Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
--------

In [85]:
analysis_schema = {
    "type": "object",
    "properties": {
        "professional_summary": {
            "type": "string",
            "description": (
                "Resumo profissional curto, baseado somente nas evidências."
            )
        },
        "strengths": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {
                        "type": "string"
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "explanation": {
                        "type": "string"
                    }
                },
                "required": [
                    "skill",
                    "evidence_ids",
                    "explanation"
                ]
            }
        },
        "gaps": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {
                        "type": "string"
                    },
                    "explanation": {
                        "type": "string"
                    }
                },
                "required": [
                    "skill",
                    "explanation"
                ]
            }
        },
        "warnings": {
            "type": "array",
            "items": {
                "type": "string"
            }
        }
    },
    "required": [
        "professional_summary",
        "strengths",
        "gaps",
        "warnings"
    ]
}


In [86]:
structured_prompt = f"""
Você é um assistente de carreira responsável por analisar
um candidato para uma vaga profissional.

O objetivo é analisar um currículo profissional.
Não confunda currículo profissional com currículo escolar
ou plano educacional.

Regras obrigatórias:

1. Use somente as evidências fornecidas.
2. Não invente experiências, cargos, datas, instituições ou habilidades.
3. Não transforme uma evidência candidata em uma competência confirmada.
4. Se não houver evidência suficiente, registre a informação em gaps.
5. Preserve o significado das experiências originais.
6. Responda em português.
7. Retorne somente o objeto JSON solicitado.
8. Use os IDs reais das evidências quando fizer uma associação.
9. Não crie IDs novos.

Descrição da vaga:
{job_description}

Análise de competências:
{evidence_context}
"""


In [87]:
structured_response = generate_with_retry(
    client=gemini_client,
    model=gemini_model_name,
    contents=structured_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": analysis_schema
    }
)


Tentativa 1/3...
Falhou: ServerError
Aguardando 20 segundos...
Tentativa 2/3...
Falhou: ServerError
Aguardando 40 segundos...
Tentativa 3/3...
Resposta recebida.


In [88]:
print(structured_response.text)


{
  "professional_summary": "Estudante de Sistemas de Informação com previsão de conclusão em 2027 e experiência acadêmica em iniciação científica voltada à avaliação de modelos de linguagem e análise de dados textuais. Possui conhecimentos práticos em Python, machine learning e processamento de linguagem natural por meio de cursos e projetos experimentais.",
  "strengths": [
    {
      "skill": "Python",
      "evidence_ids": ["chunk-0002", "chunk-0003"],
      "explanation": "Utilizou Python no desenvolvimento de assistente de estudos com PLN e possui formação complementar via curso livre de Python para análise de dados."
    },
    {
      "skill": "Machine Learning",
      "evidence_ids": ["chunk-0003"],
      "explanation": "Listada como habilidade técnica no currículo, complementada pela realização de curso livre introdutório à área."
    },
    {
      "skill": "Análise de dados",
      "evidence_ids": ["chunk-0003"],
      "explanation": "Consta como habilidade no perfil profi

In [89]:
import json

structured_analysis = json.loads(
    structured_response.text
)

print(structured_analysis.keys())


dict_keys(['professional_summary', 'strengths', 'gaps', 'warnings'])


In [90]:
assert "professional_summary" in structured_analysis
assert "strengths" in structured_analysis
assert "gaps" in structured_analysis
assert "warnings" in structured_analysis

assert isinstance(
    structured_analysis["professional_summary"],
    str
)

assert isinstance(
    structured_analysis["strengths"],
    list
)

assert isinstance(
    structured_analysis["gaps"],
    list
)

assert isinstance(
    structured_analysis["warnings"],
    list
)

print("Resposta estruturada validada.")


Resposta estruturada validada.


In [91]:
valid_chunk_ids = {
    chunk["chunk_id"]
    for chunk in chunks
}

used_evidence_ids = set()

for strength in structured_analysis["strengths"]:
    for evidence_id in strength["evidence_ids"]:
        used_evidence_ids.add(evidence_id)

invalid_ids = used_evidence_ids - valid_chunk_ids

if invalid_ids:
    print("IDs inválidos encontrados:", invalid_ids)
else:
    print("Todos os IDs de evidência são válidos.")


Todos os IDs de evidência são válidos.


In [92]:
print("RESUMO PROFISSIONAL")
print("=" * 70)
print(structured_analysis["professional_summary"])

print("\nPONTOS FORTES")
print("=" * 70)

for strength in structured_analysis["strengths"]:
    print(f"- {strength['skill']}")
    print(f"  Evidências: {strength['evidence_ids']}")
    print(f"  Explicação: {strength['explanation']}")

print("\nLACUNAS")
print("=" * 70)

for gap in structured_analysis["gaps"]:
    print(f"- {gap['skill']}")
    print(f"  {gap['explanation']}")

print("\nAVISOS")
print("=" * 70)

for warning in structured_analysis["warnings"]:
    print(f"- {warning}")


RESUMO PROFISSIONAL
Estudante de Sistemas de Informação com previsão de conclusão em 2027 e experiência acadêmica em iniciação científica voltada à avaliação de modelos de linguagem e análise de dados textuais. Possui conhecimentos práticos em Python, machine learning e processamento de linguagem natural por meio de cursos e projetos experimentais.

PONTOS FORTES
- Python
  Evidências: ['chunk-0002', 'chunk-0003']
  Explicação: Utilizou Python no desenvolvimento de assistente de estudos com PLN e possui formação complementar via curso livre de Python para análise de dados.
- Machine Learning
  Evidências: ['chunk-0003']
  Explicação: Listada como habilidade técnica no currículo, complementada pela realização de curso livre introdutório à área.
- Análise de dados
  Evidências: ['chunk-0003']
  Explicação: Consta como habilidade no perfil profissional, reforçada por curso livre específico de Python para análise de dados.
- Modelos de linguagem
  Evidências: ['chunk-0001', 'chunk-0002']
 

In [93]:
resume_schema = {
    "type": "object",
    "properties": {
        "personal_info": {
            "type": "object",
            "properties": {
                "name": {
                    "type": ["string", "null"]
                },
                "email": {
                    "type": ["string", "null"]
                },
                "phone": {
                    "type": ["string", "null"]
                },
                "location": {
                    "type": ["string", "null"]
                },
                "linkedin": {
                    "type": ["string", "null"]
                },
                "github": {
                    "type": ["string", "null"]
                },
                "portfolio": {
                    "type": ["string", "null"]
                }
            },
            "required": [
                "name",
                "email",
                "phone",
                "location",
                "linkedin",
                "github",
                "portfolio"
            ]
        },
        "objective": {
            "type": "object",
            "properties": {
                "text": {
                    "type": ["string", "null"]
                },
                "target_role": {
                    "type": ["string", "null"]
                }
            },
            "required": [
                "text",
                "target_role"
            ]
        },
        "professional_summary": {
            "type": "string"
        },
        "technical_skills": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string"
                    },
                    "category": {
                        "type": "string",
                        "enum": [
                            "language",
                            "framework",
                            "library",
                            "database",
                            "tool",
                            "platform",
                            "methodology",
                            "other"
                        ]
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "name",
                    "category",
                    "evidence_ids"
                ]
            }
        },
        "professional_experience": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "role": {
                        "type": "string"
                    },
                    "organization": {
                        "type": ["string", "null"]
                    },
                    "location": {
                        "type": ["string", "null"]
                    },
                    "start_date": {
                        "type": ["string", "null"]
                    },
                    "end_date": {
                        "type": ["string", "null"]
                    },
                    "is_current": {
                        "type": "boolean"
                    },
                    "description": {
                        "type": ["string", "null"]
                    },
                    "achievements": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "technologies": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "role",
                    "organization",
                    "location",
                    "start_date",
                    "end_date",
                    "is_current",
                    "description",
                    "achievements",
                    "technologies",
                    "evidence_ids"
                ]
            }
        },
        "academic_experience": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string"
                    },
                    "organization": {
                        "type": ["string", "null"]
                    },
                    "start_date": {
                        "type": ["string", "null"]
                    },
                    "end_date": {
                        "type": ["string", "null"]
                    },
                    "description": {
                        "type": "string"
                    },
                    "achievements": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "technologies": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "title",
                    "organization",
                    "start_date",
                    "end_date",
                    "description",
                    "achievements",
                    "technologies",
                    "evidence_ids"
                ]
            }
        },
        "education": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "course": {
                        "type": "string"
                    },
                    "institution": {
                        "type": ["string", "null"]
                    },
                    "status": {
                        "type": ["string", "null"]
                    },
                    "start_date": {
                        "type": ["string", "null"]
                    },
                    "expected_completion": {
                        "type": ["string", "null"]
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "course",
                    "institution",
                    "status",
                    "start_date",
                    "expected_completion",
                    "evidence_ids"
                ]
            }
        },
        "languages": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "language": {
                        "type": "string"
                    },
                    "level": {
                        "type": ["string", "null"]
                    },
                    "details": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "language",
                    "level",
                    "details",
                    "evidence_ids"
                ]
            }
        },
        "projects": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string"
                    },
                    "context": {
                        "type": ["string", "null"]
                    },
                    "description": {
                        "type": "string"
                    },
                    "contributions": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "technologies": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "links": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "name",
                    "context",
                    "description",
                    "contributions",
                    "technologies",
                    "links",
                    "evidence_ids"
                ]
            }
        },
        "courses_and_certifications": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string"
                    },
                    "institution": {
                        "type": ["string", "null"]
                    },
                    "completion_date": {
                        "type": ["string", "null"]
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    }
                },
                "required": [
                    "name",
                    "institution",
                    "completion_date",
                    "evidence_ids"
                ]
            }
        },
        "missing_information": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },
        "warnings": {
            "type": "array",
            "items": {
                "type": "string"
            }
        }
    },
    "required": [
        "personal_info",
        "objective",
        "professional_summary",
        "technical_skills",
        "professional_experience",
        "academic_experience",
        "education",
        "languages",
        "projects",
        "courses_and_certifications",
        "missing_information",
        "warnings"
    ]
}


In [94]:
# Contexto completo dos documentos, preservando os IDs dos chunks
candidate_context = "\n\n".join(
    [
        (
            f"ID DA EVIDÊNCIA: {chunk['chunk_id']}\n"
            f"FONTE: {chunk['source_name']}\n"
            f"TIPO: {chunk['source_type']}\n"
            f"PÁGINA: {chunk['metadata'].get('page')}\n"
            f"CONTEÚDO:\n{chunk['content']}"
        )
        for chunk in chunks
    ]
)


resume_generation_prompt = f"""
Você é um assistente especializado em estruturar currículos profissionais.

Sua tarefa é criar um currículo estruturado para um candidato, adaptado
à descrição da vaga, usando exclusivamente as informações presentes nas
evidências fornecidas.

O resultado será usado por uma aplicação chamada Pathfolio.

DESCRIÇÃO DA VAGA:
{job_description}

ANÁLISE DE COMPETÊNCIAS DA VAGA:
{json.dumps(hybrid_analysis, ensure_ascii=False, indent=2)}

ANÁLISE PROFISSIONAL ANTERIOR:
{json.dumps(structured_analysis, ensure_ascii=False, indent=2)}

EVIDÊNCIAS COMPLETAS DOS DOCUMENTOS:
{candidate_context}

REGRAS ABSOLUTAS DE FIDELIDADE:

1. Use somente informações presentes nas evidências.
2. Não invente nome, e-mail, telefone, localização, links, cargos,
   empresas, instituições, datas, tecnologias, resultados ou experiências.
3. Se uma informação não estiver disponível, use null ou uma lista vazia,
   conforme o tipo de campo.
4. Não transforme um curso em experiência profissional.
5. Não transforme uma habilidade declarada em experiência prática.
6. Não transforme um projeto acadêmico em emprego formal.
7. Não transforme uma associação semântica em competência confirmada.
8. Não crie métricas, números ou resultados que não estejam nas evidências.
9. Não crie links. Inclua links somente se estiverem escritos nas fontes.
10. Não crie datas aproximadas.
11. Preserve o significado original das experiências.
12. Todos os evidence_ids devem ser IDs reais encontrados nas evidências.
13. Cada item criado deve conter somente os IDs das evidências que realmente
    sustentam aquele item.
14. Se uma seção não possuir evidência suficiente, retorne uma lista vazia.
15. Responda em português brasileiro.
16. Retorne somente o objeto JSON definido pelo schema.
17. Não inclua Markdown, comentários ou texto fora do JSON.

REGRAS PARA OS CAMPOS:

### personal_info

Extraia somente dados pessoais explicitamente encontrados nas evidências:

- name;
- email;
- phone;
- location;
- linkedin;
- github;
- portfolio.

Se um dado não aparecer, use null.

### objective

Se houver um objetivo profissional explícito no documento, preserve-o
e adapte apenas levemente para a vaga.

Não invente um objetivo caso ele não esteja presente.

O campo target_role deve conter somente um cargo ou área explicitamente
mencionado ou claramente indicado nas evidências.

### professional_summary

Crie um resumo profissional curto, objetivo e fiel às evidências.

Não apresente uma pessoa estudante como profissional experiente se isso
não estiver comprovado.

Não mencione uma tecnologia somente porque ela aparece na descrição da vaga.

### technical_skills

Inclua habilidades técnicas encontradas nas fontes.

Classifique cada habilidade em uma destas categorias:

- language;
- framework;
- library;
- database;
- tool;
- platform;
- methodology;
- other.

Uma habilidade listada sem descrição detalhada ainda pode aparecer nesta
seção, mas não deve ser apresentada como experiência prática.

### professional_experience

Inclua somente experiências profissionais, estágios, trabalhos formais,
voluntariado profissional ou atuações equivalentes explicitamente presentes
nas evidências.

Cada item deve separar:

- role;
- organization;
- location;
- start_date;
- end_date;
- is_current;
- description;
- achievements;
- technologies;
- evidence_ids.

Se a evidência não informar um campo, use null.

### academic_experience

Use esta seção para:

- iniciação científica;
- pesquisa acadêmica;
- atividades acadêmicas relevantes;
- experiências de laboratório;
- trabalhos acadêmicos com escopo significativo.

Não coloque uma iniciação científica em professional_experience.

### education

Inclua formações acadêmicas ou técnicas.

Separe:

- course;
- institution;
- status;
- start_date;
- expected_completion.

Não invente datas.

### languages

Inclua somente idiomas explicitamente mencionados.

Se houver nível e detalhes como leitura, escrita ou conversação,
preserve essas informações.

### projects

Inclua projetos pessoais, acadêmicos, de pesquisa ou de portfólio.

Separe:

- name;
- context;
- description;
- contributions;
- technologies;
- links;
- evidence_ids.

O campo context pode indicar, por exemplo:

- projeto acadêmico;
- projeto pessoal;
- iniciação científica;
- projeto de portfólio;
- trabalho de conclusão de curso.

Use esse valor somente quando houver evidência para isso.

### courses_and_certifications

Inclua cursos, treinamentos e certificações explicitamente mencionados.

Não transforme cursos em experiências profissionais.

### missing_information

Liste informações importantes que não foram encontradas, como:

- experiência profissional formal;
- datas;
- instituição;
- links;
- resultados mensuráveis;
- informações de contato.

Não trate uma informação ausente como prova de que o candidato não possui
aquela experiência.

### warnings

Inclua alertas sobre limitações da análise, como:

- competência encontrada apenas como habilidade declarada;
- evidência semântica não confirmada;
- ausência de datas;
- ausência de informações de contato;
- ausência de experiência profissional formal.

Retorne agora o currículo estruturado conforme o schema informado.
"""


In [95]:
resume_response = generate_with_retry(
    client=gemini_client,
    model=gemini_model_name,
    contents=resume_generation_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": resume_schema
    }
)


Tentativa 1/3...
Falhou: ServerError
Aguardando 20 segundos...
Tentativa 2/3...
Resposta recebida.


In [96]:
resume_draft = json.loads(
    resume_response.text
)

print(resume_draft.keys())


dict_keys(['personal_info', 'objective', 'professional_summary', 'technical_skills', 'professional_experience', 'academic_experience', 'education', 'languages', 'projects', 'courses_and_certifications', 'missing_information', 'warnings'])


In [97]:
expected_keys = {
    "personal_info",
    "objective",
    "professional_summary",
    "technical_skills",
    "professional_experience",
    "academic_experience",
    "education",
    "languages",
    "projects",
    "courses_and_certifications",
    "missing_information",
    "warnings"
}

missing_keys = expected_keys - set(resume_draft.keys())

if missing_keys:
    print("Chaves ausentes:", missing_keys)
else:
    print("Todas as seções do novo currículo foram geradas.")


Todas as seções do novo currículo foram geradas.


## 7. Montagem do currículo estruturado

Nesta etapa, o resultado gerado pelo modelo será organizado em uma estrutura
final de currículo.

A organização será feita em Python, sem uma nova chamada ao LLM, para evitar
alterações desnecessárias nas informações já analisadas.


In [98]:
print("DADOS PESSOAIS")
print("=" * 70)

print(resume_draft["personal_info"])

print("\nOBJETIVO")
print("=" * 70)

print(resume_draft["objective"])

print("\nRESUMO PROFISSIONAL")
print("=" * 70)

print(resume_draft["professional_summary"])

print("\nHABILIDADES TÉCNICAS")
print("=" * 70)

for skill in resume_draft["technical_skills"]:
    print(f"- {skill['name']}")
    print(f"  Categoria: {skill['category']}")
    print(f"  Evidências: {skill['evidence_ids']}")

print("\nEXPERIÊNCIA PROFISSIONAL")
print("=" * 70)

for experience in resume_draft["professional_experience"]:
    print(f"\nCargo: {experience['role']}")
    print(f"Organização: {experience['organization']}")
    print(
        f"Período: {experience['start_date']} "
        f"até {experience['end_date']}"
    )
    print(f"Atualmente: {experience['is_current']}")
    print(f"Descrição: {experience['description']}")

    print("Realizações:")
    for achievement in experience["achievements"]:
        print(f"- {achievement}")

    print(f"Tecnologias: {experience['technologies']}")
    print(f"Evidências: {experience['evidence_ids']}")

print("\nEXPERIÊNCIA ACADÊMICA")
print("=" * 70)

for experience in resume_draft["academic_experience"]:
    print(f"\nTítulo: {experience['title']}")
    print(f"Organização: {experience['organization']}")
    print(
        f"Período: {experience['start_date']} "
        f"até {experience['end_date']}"
    )
    print(f"Descrição: {experience['description']}")

    print("Realizações:")
    for achievement in experience["achievements"]:
        print(f"- {achievement}")

    print(f"Tecnologias: {experience['technologies']}")
    print(f"Evidências: {experience['evidence_ids']}")

print("\nFORMAÇÃO")
print("=" * 70)

for education in resume_draft["education"]:
    print(f"\nCurso: {education['course']}")
    print(f"Instituição: {education['institution']}")
    print(f"Status: {education['status']}")
    print(f"Início: {education['start_date']}")
    print(
        f"Conclusão prevista: "
        f"{education['expected_completion']}"
    )
    print(f"Evidências: {education['evidence_ids']}")

print("\nIDIOMAS")
print("=" * 70)

for language in resume_draft["languages"]:
    print(f"\nIdioma: {language['language']}")
    print(f"Nível: {language['level']}")
    print(f"Detalhes: {language['details']}")
    print(f"Evidências: {language['evidence_ids']}")

print("\nPROJETOS")
print("=" * 70)

for project in resume_draft["projects"]:
    print(f"\nNome: {project['name']}")
    print(f"Contexto: {project['context']}")
    print(f"Descrição: {project['description']}")

    print("Contribuições:")
    for contribution in project["contributions"]:
        print(f"- {contribution}")

    print(f"Tecnologias: {project['technologies']}")
    print(f"Links: {project['links']}")
    print(f"Evidências: {project['evidence_ids']}")

print("\nCURSOS E CERTIFICAÇÕES")
print("=" * 70)

for course in resume_draft["courses_and_certifications"]:
    print(f"\nNome: {course['name']}")
    print(f"Instituição: {course['institution']}")
    print(f"Conclusão: {course['completion_date']}")
    print(f"Evidências: {course['evidence_ids']}")

print("\nINFORMAÇÕES AUSENTES")
print("=" * 70)

for item in resume_draft["missing_information"]:
    print(f"- {item}")

print("\nAVISOS")
print("=" * 70)

for warning in resume_draft["warnings"]:
    print(f"- {warning}")


DADOS PESSOAIS
{'name': 'Marina Oliveira', 'email': None, 'phone': None, 'location': None, 'linkedin': None, 'github': None, 'portfolio': None}

OBJETIVO
{'text': 'Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações Inteligentes.', 'target_role': 'Estágio em Inteligência Artificial'}

RESUMO PROFISSIONAL
Estudante de Sistemas de Informação com foco em Inteligência Artificial e experiência em iniciação científica voltada para avaliação e comparação de modelos de linguagem e análise de dados textuais. Possui projetos práticos envolvendo Python, processamento de linguagem natural e análise exploratória de dados.

HABILIDADES TÉCNICAS
- Python
  Categoria: language
  Evidências: ['chunk-0002', 'chunk-0003', 'chunk-0006']
- Machine Learning
  Categoria: methodology
  Evidências: ['chunk-0003']
- Processamento de Linguagem Natural
  Categoria: methodology
  Evidências: ['chunk-0001', 'chunk-0002', 'chunk-0003', 'chunk-0006']
- Modelos de Linguage

In [99]:
required_resume_keys = {
    "personal_info",
    "objective",
    "professional_summary",
    "technical_skills",
    "professional_experience",
    "academic_experience",
    "education",
    "languages",
    "projects",
    "courses_and_certifications",
    "missing_information",
    "warnings"
}

missing_keys = (
    required_resume_keys - set(resume_draft.keys())
)

assert not missing_keys, (
    f"Seções ausentes: {missing_keys}"
)

assert isinstance(
    resume_draft["personal_info"],
    dict
)

assert isinstance(
    resume_draft["objective"],
    dict
)

assert isinstance(
    resume_draft["professional_summary"],
    str
)

for section in [
    "technical_skills",
    "professional_experience",
    "academic_experience",
    "education",
    "languages",
    "projects",
    "courses_and_certifications",
    "missing_information",
    "warnings"
]:
    assert isinstance(
        resume_draft[section],
        list
    )

print("Estrutura do novo currículo validada.")


Estrutura do novo currículo validada.


In [100]:
valid_chunk_ids = {
    chunk["chunk_id"]
    for chunk in chunks
}

resume_evidence_ids = set()

sections_with_evidence = [
    "technical_skills",
    "professional_experience",
    "academic_experience",
    "education",
    "languages",
    "projects",
    "courses_and_certifications"
]

for section_name in sections_with_evidence:
    for item in resume_draft[section_name]:
        resume_evidence_ids.update(
            item.get("evidence_ids", [])
        )

invalid_ids = (
    resume_evidence_ids - valid_chunk_ids
)

assert not invalid_ids, (
    f"IDs de evidência inválidos: {invalid_ids}"
)

print("Todos os IDs de evidência são válidos.")


Todos os IDs de evidência são válidos.


In [101]:
for experience in resume_draft["professional_experience"]:
    assert experience["evidence_ids"], (
        "Experiência profissional sem evidência."
    )

for experience in resume_draft["academic_experience"]:
    assert experience["evidence_ids"], (
        "Experiência acadêmica sem evidência."
    )

for project in resume_draft["projects"]:
    assert project["evidence_ids"], (
        "Projeto sem evidência."
    )

print("Validação de fidelidade aprovada.")


Validação de fidelidade aprovada.


In [102]:
from IPython.display import Markdown, display


def render_structured_resume_markdown(resume: dict) -> str:
    """
    Converte o resume_draft estruturado em uma prévia Markdown.

    Os evidence_ids permanecem apenas no JSON interno e não aparecem
    no currículo visualizado pelo candidato.
    """

    lines = []

    personal_info = resume.get("personal_info", {})

    name = personal_info.get("name")

    if name:
        lines.append(f"# {name}")
    else:
        lines.append("# Currículo")

    contact_parts = []

    for field in ["email", "phone", "location"]:
        value = personal_info.get(field)

        if value:
            contact_parts.append(str(value))

    for field, label in [
        ("linkedin", "LinkedIn"),
        ("github", "GitHub"),
        ("portfolio", "Portfólio")
    ]:
        value = personal_info.get(field)

        if value:
            contact_parts.append(f"[{label}]({value})")

    if contact_parts:
        lines.append("\n" + " · ".join(contact_parts))

    objective = resume.get("objective", {})
    objective_text = objective.get("text")

    if objective_text:
        lines.append("\n## Objetivo")
        lines.append(objective_text)

    professional_summary = resume.get(
        "professional_summary",
        ""
    )

    if professional_summary:
        lines.append("\n## Resumo profissional")
        lines.append(professional_summary)

    technical_skills = resume.get(
        "technical_skills",
        []
    )

    if technical_skills:
        lines.append("\n## Habilidades técnicas")

        skills_by_category = {}

        for skill in technical_skills:
            category = skill.get("category", "other")
            name = skill.get("name")

            if name:
                skills_by_category.setdefault(
                    category,
                    []
                ).append(name)

        for category, skills in skills_by_category.items():
            category_label = category.replace(
                "_",
                " "
            ).title()

            lines.append(
                f"\n**{category_label}:** "
                + ", ".join(skills)
            )

    professional_experience = resume.get(
        "professional_experience",
        []
    )

    if professional_experience:
        lines.append("\n## Experiência profissional")

        for experience in professional_experience:
            role = experience.get(
                "role",
                "Cargo não informado"
            )

            organization = experience.get(
                "organization"
            )

            heading = role

            if organization:
                heading += f" — {organization}"

            lines.append(f"\n### {heading}")

            period = []

            if experience.get("start_date"):
                period.append(
                    experience["start_date"]
                )

            if experience.get("end_date"):
                period.append(
                    experience["end_date"]
                )

            if period:
                lines.append(
                    f"**Período:** {' – '.join(period)}"
                )

            if experience.get("location"):
                lines.append(
                    f"**Localização:** "
                    f"{experience['location']}"
                )

            if experience.get("description"):
                lines.append(
                    experience["description"]
                )

            for achievement in experience.get(
                "achievements",
                []
            ):
                lines.append(f"- {achievement}")

            technologies = experience.get(
                "technologies",
                []
            )

            if technologies:
                lines.append(
                    "\n**Tecnologias:** "
                    + ", ".join(technologies)
                )

    academic_experience = resume.get(
        "academic_experience",
        []
    )

    if academic_experience:
        lines.append("\n## Experiência acadêmica")

        for experience in academic_experience:
            title = experience.get(
                "title",
                "Experiência acadêmica"
            )

            organization = experience.get(
                "organization"
            )

            heading = title

            if organization:
                heading += f" — {organization}"

            lines.append(f"\n### {heading}")

            period = []

            if experience.get("start_date"):
                period.append(
                    experience["start_date"]
                )

            if experience.get("end_date"):
                period.append(
                    experience["end_date"]
                )

            if period:
                lines.append(
                    f"**Período:** {' – '.join(period)}"
                )

            if experience.get("description"):
                lines.append(
                    experience["description"]
                )

            for achievement in experience.get(
                "achievements",
                []
            ):
                lines.append(f"- {achievement}")

            technologies = experience.get(
                "technologies",
                []
            )

            if technologies:
                lines.append(
                    "\n**Tecnologias:** "
                    + ", ".join(technologies)
                )

    education = resume.get(
        "education",
        []
    )

    if education:
        lines.append("\n## Formação acadêmica")

        for item in education:
            course = item.get(
                "course",
                "Formação não informada"
            )

            institution = item.get(
                "institution"
            )

            heading = course

            if institution:
                heading += f" — {institution}"

            lines.append(f"\n### {heading}")

            details = []

            if item.get("status"):
                details.append(
                    f"Status: {item['status']}"
                )

            if item.get("start_date"):
                details.append(
                    f"Início: {item['start_date']}"
                )

            if item.get("expected_completion"):
                details.append(
                    "Conclusão prevista: "
                    f"{item['expected_completion']}"
                )

            if details:
                lines.append(
                    " · ".join(details)
                )

    languages = resume.get(
        "languages",
        []
    )

    if languages:
        lines.append("\n## Idiomas")

        for item in languages:
            language = item.get(
                "language",
                "Idioma não informado"
            )

            language_line = f"- **{language}**"

            if item.get("level"):
                language_line += (
                    f": {item['level']}"
                )

            details = item.get(
                "details",
                []
            )

            if details:
                language_line += (
                    f" ({', '.join(details)})"
                )

            lines.append(language_line)

    projects = resume.get(
        "projects",
        []
    )

    if projects:
        lines.append("\n## Projetos")

        for project in projects:
            name = project.get(
                "name",
                "Projeto sem nome"
            )

            context = project.get(
                "context"
            )

            heading = name

            if context:
                heading += f" — {context}"

            lines.append(f"\n### {heading}")

            if project.get("description"):
                lines.append(
                    project["description"]
                )

            for contribution in project.get(
                "contributions",
                []
            ):
                lines.append(f"- {contribution}")

            technologies = project.get(
                "technologies",
                []
            )

            if technologies:
                lines.append(
                    "\n**Tecnologias:** "
                    + ", ".join(technologies)
                )

            links = project.get(
                "links",
                []
            )

            if links:
                lines.append(
                    "\n**Links:** "
                    + " · ".join(links)
                )

    courses = resume.get(
        "courses_and_certifications",
        []
    )

    if courses:
        lines.append(
            "\n## Cursos e certificações"
        )

        for course in courses:
            course_line = (
                f"- **{course.get('name', 'Curso não informado')}**"
            )

            details = []

            if course.get("institution"):
                details.append(
                    course["institution"]
                )

            if course.get("completion_date"):
                details.append(
                    course["completion_date"]
                )

            if details:
                course_line += (
                    f" — {' · '.join(details)}"
                )

            lines.append(course_line)

    return "\n".join(lines).strip()


resume_markdown = render_structured_resume_markdown(
    resume_draft
)

assert isinstance(resume_markdown, str)
assert resume_markdown.strip()

if resume_draft["professional_summary"]:
    assert "## Resumo profissional" in resume_markdown

if resume_draft["professional_experience"]:
    assert "## Experiência profissional" in resume_markdown

if resume_draft["academic_experience"]:
    assert "## Experiência acadêmica" in resume_markdown

if resume_draft["education"]:
    assert "## Formação acadêmica" in resume_markdown

if resume_draft["languages"]:
    assert "## Idiomas" in resume_markdown

if resume_draft["projects"]:
    assert "## Projetos" in resume_markdown

if resume_draft["courses_and_certifications"]:
    assert (
        "## Cursos e certificações"
        in resume_markdown
    )

display(Markdown(resume_markdown))

with open(
    "pathfolio_structured_resume_preview.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(resume_markdown)

print(
    "\nRenderização do currículo estruturado aprovada."
)


# Marina Oliveira

## Objetivo
Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações Inteligentes.

## Resumo profissional
Estudante de Sistemas de Informação com foco em Inteligência Artificial e experiência em iniciação científica voltada para avaliação e comparação de modelos de linguagem e análise de dados textuais. Possui projetos práticos envolvendo Python, processamento de linguagem natural e análise exploratória de dados.

## Habilidades técnicas

**Language:** Python, SQL básico, JavaScript

**Methodology:** Machine Learning, Processamento de Linguagem Natural, Análise de Dados

**Other:** Modelos de Linguagem

**Tool:** Git

**Platform:** GitHub

**Library:** Pandas

**Framework:** React

## Experiência acadêmica

### Projeto de iniciação científica — Avaliação de modelos de linguagem — Universidade Exemplo
Comparação de modelos de inteligência artificial em tarefas de análise textual e correção de redações.
- Comparação de modelos de inteligência artificial em tarefas de análise textual.
- Organização e tratamento de uma base de dados com exemplos de textos.
- Análise de métricas e interpretação dos resultados experimentais.
- Documentação da metodologia e apresentação dos resultados.

**Tecnologias:** Modelos de Linguagem, Inteligência Artificial

## Formação acadêmica

### Graduação em Sistemas de Informação — Universidade Exemplo
Status: Em andamento · Conclusão prevista: 2027

## Projetos

### Assistente de estudos com processamento de linguagem natural — Projeto experimental
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo.
- Geração de perguntas de revisão e resumos a partir de materiais de estudo
- Utilização de Python, processamento de texto e modelos de linguagem

**Tecnologias:** Python, Processamento de texto, Modelos de linguagem

### Classificação de feedbacks acadêmicos — Projeto experimental
Investigação de técnicas de processamento de linguagem natural para organizar feedbacks acadêmicos por tema e prioridade, facilitando a análise por professores.
- Coleta e limpeza de exemplos textuais fictícios
- Definição de categorias para classificação dos feedbacks
- Criação de uma rotina de pré-processamento em Python
- Avaliação qualitativa dos resultados e análise de erros
- Registro das decisões e limitações do experimento

**Tecnologias:** Python, Pandas, Git

### Painel de acompanhamento de estudos — Aplicação web fictícia
Aplicação web criada para organizar metas de estudo, registrar sessões e sugerir revisões com base no histórico do usuário.
- Levantamento dos requisitos principais da aplicação
- Modelagem das entidades de tarefas, sessões e metas
- Criação de componentes de interface para cadastro e acompanhamento
- Implementação de validações para evitar registros incompletos
- Documentação do fluxo principal e das decisões do projeto

**Tecnologias:** JavaScript, React, Git

## Cursos e certificações
- **Introdução a Machine Learning — curso livre**
- **Python para análise de dados — curso livre**
- **Participação em grupo de estudos sobre Inteligência Artificial Generativa**


Renderização do currículo estruturado aprovada.


## 8. Prévia legível do currículo

Nesta etapa, o currículo estruturado será convertido em Markdown para
visualização e revisão.

A conversão será feita em Python, sem uma nova chamada ao modelo.
